In [1]:
import torch

# 수업 토큰 임베딩 (난이도, 과제량, 흥미도)
inputs = torch.tensor(
    [[0.80, 0.70, 0.60], # 회귀분석
     [0.90, 0.80, 0.75], # 수통2
     [0.75, 0.85, 0.70], # 자료구조
     [0.60, 0.65, 0.50], # 표출
     [0.85, 0.90, 0.80], # 기학
     [0.20, 0.30, 0.95]] # 사랑과결혼
)

# 기학 관점에서 "어떤 수업이 나랑 비슷한가?"
query = inputs[4]  # 기학

attn_scores = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores[i] = torch.dot(x_i, query)

print("어텐션 점수:", attn_scores)

attn_weights = torch.softmax(attn_scores, dim=0)
print("어텐션 가중치:", attn_weights)
print("합:", attn_weights.sum())

context_vec = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec += attn_weights[i] * x_i

print("\n기학 관점 문맥 벡터:", context_vec)

어텐션 점수: tensor([1.7900, 2.0850, 1.9625, 1.4950, 2.1725, 1.2000])
어텐션 가중치: tensor([0.1588, 0.2133, 0.1887, 0.1183, 0.2328, 0.0880])
합: tensor(1.)

기학 관점 문맥 벡터: tensor([0.7471, 0.7551, 0.7164])


In [2]:
# 모든 수업 쌍에 대해 어텐션 점수 한번에 계산
attn_scores_all = inputs @ inputs.T
print("전체 어텐션 점수 행렬:\n", attn_scores_all)

# 소프트맥스 정규화
attn_weights_all = torch.softmax(attn_scores_all, dim=-1)
print("\n전체 어텐션 가중치 행렬:\n", attn_weights_all)
print("\n각 행의 합:", attn_weights_all.sum(dim=-1))

# 모든 문맥 벡터 한번에 계산
all_context_vecs = attn_weights_all @ inputs
print("\n모든 문맥 벡터:\n", all_context_vecs)
print("\n기학 문맥 벡터(4번째 행):", all_context_vecs[4])
print("이전에 계산한 값:", context_vec)

전체 어텐션 점수 행렬:
 tensor([[1.4900, 1.7300, 1.6150, 1.2350, 1.7900, 0.9400],
        [1.7300, 2.0125, 1.8800, 1.4350, 2.0850, 1.1325],
        [1.6150, 1.8800, 1.7750, 1.3525, 1.9625, 1.0700],
        [1.2350, 1.4350, 1.3525, 1.0325, 1.4950, 0.7900],
        [1.7900, 2.0850, 1.9625, 1.4950, 2.1725, 1.2000],
        [0.9400, 1.1325, 1.0700, 0.7900, 1.2000, 1.0325]])

전체 어텐션 가중치 행렬:
 tensor([[0.1638, 0.2082, 0.1856, 0.1269, 0.2211, 0.0945],
        [0.1611, 0.2136, 0.1871, 0.1199, 0.2297, 0.0886],
        [0.1602, 0.2088, 0.1880, 0.1232, 0.2268, 0.0929],
        [0.1639, 0.2002, 0.1844, 0.1339, 0.2126, 0.1050],
        [0.1588, 0.2133, 0.1887, 0.1183, 0.2328, 0.0880],
        [0.1514, 0.1835, 0.1724, 0.1303, 0.1963, 0.1661]])

각 행의 합: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

모든 문맥 벡터:
 tensor([[0.7405, 0.7487, 0.7144],
        [0.7463, 0.7539, 0.7157],
        [0.7424, 0.7511, 0.7157],
        [0.7316, 0.7415, 0.7144],
        [0.7471, 0.7551, 0.7164],
        [0.6939, 0.710

In [3]:
import torch.nn as nn

torch.manual_seed(123)

d_in = inputs.shape[1]  # 3
d_out = 2

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        queries = x @ self.W_query
        keys    = x @ self.W_key
        values  = x @ self.W_value

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.3700, 1.0306],
        [0.3720, 1.0343],
        [0.3713, 1.0330],
        [0.3682, 1.0271],
        [0.3727, 1.0356],
        [0.3672, 1.0253]], grad_fn=<MmBackward0>)


In [4]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0686,  0.1460],
        [-0.0688,  0.1455],
        [-0.0687,  0.1456],
        [-0.0683,  0.1464],
        [-0.0689,  0.1453],
        [-0.0679,  0.1471]], grad_fn=<MmBackward0>)


In [5]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

# 배치 테스트 (입력 2개)
torch.manual_seed(123)
batch = torch.stack([inputs, inputs], dim=0)
print("배치 shape:", batch.shape)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, dropout=0.0)
context_vecs = ca(batch)
print("출력 shape:", context_vecs.shape)
print(context_vecs)

배치 shape: torch.Size([2, 6, 3])
출력 shape: torch.Size([2, 6, 2])
tensor([[[-0.7644, -0.1722],
         [-0.8236, -0.1716],
         [-0.8202, -0.1793],
         [-0.7734, -0.1760],
         [-0.8009, -0.1790],
         [-0.7388, -0.1191]],

        [[-0.7644, -0.1722],
         [-0.8236, -0.1716],
         [-0.8202, -0.1793],
         [-0.7734, -0.1760],
         [-0.8009, -0.1790],
         [-0.7388, -0.1191]]], grad_fn=<UnsafeViewBackward0>)


In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out은 num_heads로 나누어 떨어져야 합니다"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        # 멀티 헤드로 분할
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 헤드 결합
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec

# 테스트
torch.manual_seed(123)
mha = MultiHeadAttention(d_in, d_out=4, context_length=batch.shape[1],
                         dropout=0.0, num_heads=2)
context_vecs = mha(batch)
print("출력 shape:", context_vecs.shape)
print(context_vecs)

출력 shape: torch.Size([2, 6, 4])
tensor([[[-0.0749,  0.3196, -0.0699, -0.2840],
         [-0.0844,  0.3323, -0.0710, -0.2800],
         [-0.0874,  0.3356, -0.0705, -0.2753],
         [-0.0777,  0.3273, -0.0698, -0.2819],
         [-0.0830,  0.3332, -0.0701, -0.2782],
         [-0.0508,  0.3336, -0.0723, -0.3260]],

        [[-0.0749,  0.3196, -0.0699, -0.2840],
         [-0.0844,  0.3323, -0.0710, -0.2800],
         [-0.0874,  0.3356, -0.0705, -0.2753],
         [-0.0777,  0.3273, -0.0698, -0.2819],
         [-0.0830,  0.3332, -0.0701, -0.2782],
         [-0.0508,  0.3336, -0.0723, -0.3260]]], grad_fn=<ViewBackward0>)
